# 00 — Setup

First notebook in the Foundry harness sequence. This one does not call any LLM — it just gets the repo, dependencies, and the CodeGuard rule corpus in place, and captures your OpenAI key for later notebooks.

Run cells top to bottom. If you're running this locally (not in Colab) inside an already-cloned repo, the clone cell below detects that and skips itself.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/harshamore/FoundryHarnessDC.git"
REPO_DIR = "FoundryHarnessDC"
PROJECT_SUBDIR = "langchain"  # this project lives in a subdirectory of the repo

cwd = Path.cwd()
if (cwd / "pyproject.toml").exists():
    # Already inside the project directory (e.g. running locally from repo root).
    project_path = cwd
elif (cwd.parent / "pyproject.toml").exists():
    # Running from notebooks/ inside an already-cloned checkout.
    project_path = cwd.parent
elif (cwd / REPO_DIR).exists():
    project_path = cwd / REPO_DIR / PROJECT_SUBDIR
else:
    !git clone --quiet {REPO_URL}
    project_path = cwd / REPO_DIR / PROJECT_SUBDIR

os.chdir(project_path)
print(f"Working directory: {os.getcwd()}")

In [ ]:
%pip install --quiet -e ".[dev]"

## Fetch the CodeGuard rule corpus

Clones `cosai-oasis/project-codeguard` at a pinned commit and vendors `sources/rules/{core,owasp}` into `data/codeguard/rules/`. This has nothing to do with the Claude Code plugin on your laptop — Colab can't see that, so the harness fetches its own copy. See `docs/CODEGUARD_INTEGRATION.md`.

In [ ]:
!python scripts/fetch_codeguard_rules.py

In [ ]:
from pathlib import Path

core = list(Path("data/codeguard/rules/core").glob("*.md"))
owasp = list(Path("data/codeguard/rules/owasp").glob("*.md"))
print(f"core: {len(core)} rules, owasp: {len(owasp)} rules")
assert len(core) > 0 and len(owasp) > 0

## OpenAI API key

Entered interactively via `getpass` — never written to a file, never committed. Later notebooks (02 onward) read this from the environment. This notebook itself makes no OpenAI calls.

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
print("OPENAI_API_KEY is set:", bool(os.environ.get("OPENAI_API_KEY")))

## Sanity check

Import the substrate modules and confirm they load with no errors — proves the install worked before moving to `01_substrate.ipynb`.

In [ ]:
from foundry.substrate.budget import BudgetGovernor
from foundry.substrate.db import connect
from foundry.substrate.finding_store import FindingStore, fingerprint
from foundry.substrate.work_queue import WorkQueue

print("Substrate modules import cleanly. Continue to 01_substrate.ipynb.")